# 03 — Time-Series Profiling
**Purpose:** Determine the time window and sampling frequency for every signal.

This is critical for ZORO because:
- A model trained on 3 months behaves differently from one trained on 3.5 years
- Defrost cycle detection requires 20-second data; 15-minute resolution misses it
- Some signals are commissioning snapshots (6 rows) — useless for modeling

**This notebook answers:**
- When does each signal start and end?
- What is the sampling interval (~20s? ~5min? ~1hr?)
- Which signals are ongoing vs one-off snapshots?
- Is the data live (current) or historical?

In [ ]:
import sys, csv, statistics
from datetime import datetime
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from zoro_eda.config import load_config
from zoro_eda.paths import resolve_paths
from zoro_eda.csv_io import read_head, find_column, parse_timestamp, get_last_timestamp

cfg = load_config()
paths = resolve_paths(cfg=cfg)
print("Libraries loaded.")

## Step 1: Extract start timestamp from file head

We read the first 30 rows to get the start time AND compute intervals
between consecutive timestamps (→ sampling frequency).

**Why only 30 rows?** That's enough to compute a stable median interval
without reading more than a few KB from a potentially 300 MB file.

In [ ]:
def profile_file_head(fpath: Path, n_rows: int = 30):
    header, data_rows = read_head(fpath, n_rows=n_rows)
    time_col = find_column(header, "_time")
    
    timestamps = []
    for row in data_rows:
        if time_col >= 0 and time_col < len(row):
            dt = parse_timestamp(row[time_col])
            if dt:
                timestamps.append(dt)
    
    if not timestamps:
        return None, None
    
    start = timestamps[0]
    intervals = [
        (timestamps[i] - timestamps[i-1]).total_seconds()
        for i in range(1, len(timestamps))
        if 0 < (timestamps[i] - timestamps[i-1]).total_seconds() < 86_400
    ]
    median_interval = round(statistics.median(intervals), 1) if intervals else None
    return start, median_interval

# Test on battery SOC
fp = paths.raw_data / "greal_BatterieLadeZustand.csv"
start, interval = profile_file_head(fp)
print(f"Signal    : greal_BatterieLadeZustand")
print(f"Start UTC : {start}")
print(f"Interval  : {interval}s ({interval/60:.1f} min)")

## Step 2: Extract end timestamp from file tail

For a 300 MB file we can't read everything — but we can seek to the
last 4 KB and parse the last valid timestamp from there.

`get_last_timestamp()` in our shared library handles this, and uses
the header's column index (not a hardcoded position) for robustness.

In [ ]:
end = get_last_timestamp(fp)
duration_days = (end - start).total_seconds() / 86_400

print(f"End UTC   : {end}")
print(f"Duration  : {duration_days:.1f} days ({duration_days/365:.1f} years)")
print()
print("Data is LIVE — runs to within days of today's date!")

## Step 3: Classify the interval

We map median interval in seconds to a human-readable label:

| Seconds | Label | Typical signals |
|---|---|---|
| ≤7 | ~5s | Battery power electronics |
| ≤22 | ~20s | Core BMS — temperatures, power, energy |
| ≤65 | ~1min | Medium frequency |
| ≤310 | ~5min | HVAC aggregates, HP flow temp |
| ≤3700 | ~1hr | Setpoint files (only write on change) |

In [ ]:
def classify_interval(seconds):
    if seconds is None: return "unknown"
    if seconds <= 7:    return "~5s"
    if seconds <= 22:   return "~20s"
    if seconds <= 65:   return "~1min"
    if seconds <= 310:  return "~5min"
    if seconds <= 3_700: return "~1hr"
    return f"~{round(seconds/60)}min"

# Profile all 233 files
csv_files = sorted([f for f in paths.raw_data.iterdir()
                   if f.is_file() and f.suffix.lower() == ".csv"])

profiles = []
for fpath in csv_files:
    start, interval = profile_file_head(fpath)
    end = get_last_timestamp(fpath)
    dur = round((end - start).total_seconds()/86_400, 1) if (start and end and end > start) else None
    profiles.append({
        "file": fpath.name, "start": start, "end": end,
        "duration_days": dur,
        "median_interval_sec": interval,
        "interval_label": classify_interval(interval)
    })

from collections import Counter
print("Sampling interval distribution:")
for label, count in Counter(p["interval_label"] for p in profiles).most_common():
    print(f"  {label:12s}  {count:3d} signals")

## Step 4: Two populations of signals

Sorting by start date reveals two distinct groups:

In [ ]:
starts = sorted([p for p in profiles if p["start"]], key=lambda p: p["start"])

bms_signals     = [p for p in profiles if p["start"] and p["start"].isoformat() < "2024"]
weather_signals = [p for p in profiles if p["start"] and p["start"].isoformat() >= "2024"]

print(f"BMS operational signals (start < 2024) : {len(bms_signals)} files")
print(f"  Earliest: {bms_signals[0]['start'].isoformat()[:10]}")
print(f"  Latest end: {max(p['end'] for p in bms_signals if p['end']).isoformat()[:10]}")

print(f"\nWeather/forecast signals (start 2024+) : {len(weather_signals)} files")
for p in weather_signals:
    print(f"  {p['file']}")

## Step 5: Identify commissioning snapshots

Files with very short duration (≤7 days) are one-time exports from system
setup in December 2022. They are NOT ongoing time series.

In [ ]:
snapshot_files = [p for p in profiles
                  if p["duration_days"] is not None and p["duration_days"] < 7]

print(f"Short-lived files (<7 days): {len(snapshot_files)}")
print()
for p in sorted(snapshot_files, key=lambda x: x["duration_days"] or 0):
    print(f"  {p['file']:<45} {p['duration_days']:.1f} days  start={str(p['start'])[:10] if p['start'] else 'n/a'}")

print("\nThese are commissioning snapshots — exclude from modeling.")

## Key findings

| Finding | Value |
|---|---|
| BMS data start | December 2022 |
| Data end | May 2026 (live!) |
| Median duration | 1,254 days (~3.5 years) |
| Dominant interval | ~20 seconds (169 signals) |
| Setpoint files | ~1 hour (28 files — sparse by design) |
| Commissioning snapshots | 15 files (≤8 rows each) |

**The data is live** — it runs to within days of today. This is operational
data, not historical research data. Any model we build will be working
with current information.

**Next:** `04_signal_classification.ipynb` — what does each signal actually measure?